# 混合语义分析A/B测试结果分析

本notebook用于分析三种方案的对比结果。

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 设置样式
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 加载数据

In [ ]:
# 加载对比报告
with open('../results/comparison.json', encoding='utf-8') as f:
    report = json.load(f)

# 提取数据
methods = report['comparison']['methods']
metadata = report['metadata']

# 合并数据
data = []
for method in methods:
    method_name = method['name']
    
    # 查找对应的元数据
    meta = next((m for m in metadata if m['method'] == method_name), {})
    
    data.append({
        '方案': method_name,
        '精确率': method['precision'],
        '召回率': method['recall'],
        'F1': method['f1'],
        '实体数': method['entity_count'],
        '耗时(秒)': meta.get('time_seconds', 0),
        'Token消耗': meta.get('token_count', 0)
    })

df = pd.DataFrame(data)
df

## 2. 质量对比

In [ ]:
# 绘制精确率、召回率、F1对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['精确率', '召回率', 'F1']
for i, metric in enumerate(metrics):
    ax = axes[i]
    df.plot(x='方案', y=metric, kind='bar', ax=ax, legend=False)
    ax.set_title(f'{metric}对比')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    
    # 添加数值标签
    for j, v in enumerate(df[metric]):
        ax.text(j, v + 0.02, f'{v:.2%}', ha='center')

plt.tight_layout()
plt.savefig('../results/quality_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. 成本对比

In [ ]:
# 绘制耗时和Token消耗对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 耗时
ax = axes[0]
df.plot(x='方案', y='耗时(秒)', kind='bar', ax=ax, legend=False, color='skyblue')
ax.set_title('耗时对比')
ax.set_ylabel('耗时(秒)')
for i, v in enumerate(df['耗时(秒)']):
    ax.text(i, v + 0.2, f'{v:.2f}s', ha='center')

# Token消耗
ax = axes[1]
df.plot(x='方案', y='Token消耗', kind='bar', ax=ax, legend=False, color='salmon')
ax.set_title('Token消耗对比')
ax.set_ylabel('Token数')
for i, v in enumerate(df['Token消耗']):
    ax.text(i, v + 50, f'{v}', ha='center')

plt.tight_layout()
plt.savefig('../results/cost_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. 成本-效果权衡分析

In [ ]:
# 散点图: Token消耗 vs F1
plt.figure(figsize=(10, 6))
plt.scatter(df['Token消耗'], df['F1'], s=200, alpha=0.6)

# 添加标签
for i, row in df.iterrows():
    plt.annotate(
        row['方案'],
        (row['Token消耗'], row['F1']),
        xytext=(10, 10),
        textcoords='offset points',
        fontsize=12
    )

plt.xlabel('Token消耗', fontsize=12)
plt.ylabel('F1分数', fontsize=12)
plt.title('成本-效果权衡分析', fontsize=14)
plt.grid(True, alpha=0.3)

plt.savefig('../results/cost_effectiveness.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 节省率计算

In [ ]:
# 以纯LLM为基准计算节省率
baseline = df[df['方案'] == 'method_c_llm'].iloc[0]

savings = []
for _, row in df.iterrows():
    if row['方案'] == 'method_c_llm':
        savings.append({
            '方案': row['方案'],
            'Token节省率': '0%',
            '时间节省率': '0%',
            'F1损失': '0%'
        })
    else:
        token_saving = (baseline['Token消耗'] - row['Token消耗']) / baseline['Token消耗'] * 100
        time_saving = (baseline['耗时(秒)'] - row['耗时(秒)']) / baseline['耗时(秒)'] * 100
        f1_loss = (baseline['F1'] - row['F1']) / baseline['F1'] * 100
        
        savings.append({
            '方案': row['方案'],
            'Token节省率': f'{token_saving:.1f}%',
            '时间节省率': f'{time_saving:.1f}%',
            'F1损失': f'{f1_loss:.1f}%'
        })

savings_df = pd.DataFrame(savings)
savings_df

## 6. 结论

根据以上分析,可以得出:

1. **方案A(纯本地)**: Token成本为0,但准确率较低(适合粗筛)
2. **方案B(混合)**: 在成本和效果之间取得最佳平衡,推荐用于大规模标注
3. **方案C(纯LLM)**: 准确率最高但成本最高,适合高质量要求场景

**推荐策略**: 使用方案B(混合)进行批量标注,对关键章节使用方案C(纯LLM)。